# detach-clone-snapshot — worked example 3: Log per-step gradient snapshots from a 2D parameter

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Beyond the parameter itself, you often want to record the *gradient* at each step. `p.grad` is a live tensor that gets overwritten (or zeroed) every iteration, so appending it directly aliases the final gradient. `p.grad.detach().clone()` freezes a graph-free, independently-stored copy of the gradient at that step.

## Worked solution

**Goal:** minimize a quadratic in a length-3 vector and keep an honest history of the gradient at each step.

1. **Vector parameter.** `p` is a length-3 leaf with `requires_grad=True`, target is a fixed vector.
2. **Loss + backward.** `loss = ((p - target)**2).sum()`; `loss.backward()` populates `p.grad` (which here equals `2*(p - target)`).
3. **Snapshot the gradient.** `grad_hist.append(p.grad.detach().clone())` copies the current gradient into private storage. Without `clone()`, the next step's `zero_()`/overwrite would corrupt every stored entry.
4. **Update + zero.** Under `no_grad()`, step `p`, then `p.grad.zero_()` to clear for the next backward. This zeroing is exactly what would wipe an aliased history.
5. **Why detach matters here too.** `p.grad` itself has `requires_grad=False`, but calling `detach()` is harmless and keeps the snapshot idiom uniform; the load-bearing op is `clone()` for storage independence.
6. **Verify.** Gradient magnitudes shrink toward zero across steps, proving each snapshot captured a distinct moment rather than the final (near-zero) gradient.

In [ ]:
t.manual_seed(0)

def grad_history(steps=4, lr=0.1):
    target = t.tensor([1.0, -2.0, 0.5])
    p = t.zeros(3, requires_grad=True)
    grad_hist = []
    for _ in range(steps):
        loss = ((p - target) ** 2).sum()
        loss.backward()
        grad_hist.append(p.grad.detach().clone())
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
    return t.stack(grad_hist)

hist = grad_history()
norms = hist.norm(dim=1)
print("grad norms per step:", [round(x, 4) for x in norms.tolist()])
print("monotonically decreasing:", bool((norms[1:] < norms[:-1]).all()))